# DADT Midterm Dataset Cleaning Notebook #

## Load & inspect

Load the raw resale flat dataset and inspect its structure.

In [559]:
import pandas as pd

df = pd.read_csv("ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218676 entries, 0 to 218675
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                218676 non-null  object 
 1   town                 218676 non-null  object 
 2   flat_type            218676 non-null  object 
 3   block                218676 non-null  object 
 4   street_name          218676 non-null  object 
 5   storey_range         218676 non-null  object 
 6   floor_area_sqm       218676 non-null  float64
 7   flat_model           218676 non-null  object 
 8   lease_commence_date  218676 non-null  int64  
 9   remaining_lease      218676 non-null  object 
 10  resale_price         218676 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 18.4+ MB


The dataset has 218,676 rows, 11 columns. Dates are stored as strings (month), and lease_commence_date is an integer year. Have a quick look to check:

In [561]:
df.head(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


## Canonicalise text columns

Standardize text fields to avoid duplicates caused by inconsistent casing or whitespace.

In [564]:
text_cols = [
    "town", "street_name", "flat_type",
    "flat_model", "storey_range", "block"
]

for col in text_cols:
    df[col] = df[col].str.strip().str.upper()

Ensures stable dimension tables and prevents duplicate lookup entries

## Clean date

Convert month strings into proper datetime objects.Drop rows with invalid dates.

In [568]:
df["transaction_date"] = pd.to_datetime(df["month"], errors="coerce")
df = df.dropna(subset=["transaction_date"])

## Enforce numeric sanity

Remove invalid or nonsensical records.

In [571]:
print(df.columns)

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'remaining_lease', 'resale_price', 'transaction_date'],
      dtype='object')


In [572]:
df = df[
    (df["resale_price"] > 0) &
    (df["floor_area_sqm"] > 0) &
    (df["lease_commence_date"] >= 1960)
]

## Drop Rows Missing Key Dimensions

Ensure all dimension attributes are present.

In [575]:
required = [
    "town", "street_name", "block",
    "flat_type", "flat_model", "storey_range"
]

df = df.dropna(subset=required)

## Dimension Uniqueness Check

In [577]:
df["town"].nunique() 
df["flat_model"].nunique() 
df["storey_range"].unique()

array(['10 TO 12', '01 TO 03', '04 TO 06', '07 TO 09', '13 TO 15',
       '19 TO 21', '22 TO 24', '16 TO 18', '34 TO 36', '28 TO 30',
       '37 TO 39', '49 TO 51', '25 TO 27', '40 TO 42', '31 TO 33',
       '46 TO 48', '43 TO 45'], dtype=object)

Number of unique towns and flat models. <br>
Storey ranges follow a consistent "NN TO NN" pattern.

## Storey range split

Convert textual ranges into numeric columns. Enables numerical filtering and analysis (e.g., flats on higher floors).

In [581]:
df[["min_floor", "max_floor"]] = (
    df["storey_range"]
    .str.extract(r"(\d+)\s+TO\s+(\d+)")
    .astype(int)
)

## Rename & Drop Columns

The 'lease_commence_date' only shows year value, Rename lease_commence_date → lease_commence_year to match SQL schema.

In [584]:
df = df.rename(columns={
    "lease_commence_date": "lease_commence_year"
})

Drop remaining_lease (redundant, derived from lease year).

In [586]:
df = df.drop(columns=["remaining_lease"])

In [587]:
print(df.columns)

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_year', 'resale_price',
       'transaction_date', 'min_floor', 'max_floor'],
      dtype='object')


## Add Transaction Year & Month

Extract year and month for easier grouping and trend analysis to supports time‑series queries in SQL.

In [590]:
df["transaction_year"] = df["transaction_date"].dt.year 
df["transaction_month"] = df["transaction_date"].dt.month

## Sampling for Performance

Reduce dataset size for faster execution in Coursera environment.

In [593]:
df = df.sample(n=50000, random_state=42).sort_index() 
print("Original rows:", 218676) 
print("After cleaning:", len(df))

Original rows: 218676
After cleaning: 50000


Random sample of 50,000 rows, then sorted back by original index to preserve chronological order.

## Final Dataset Preview

In [596]:
df.head(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_year,resale_price,transaction_date,min_floor,max_floor,transaction_year,transaction_month
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,NEW GENERATION,1980,265000.0,2017-01-01,1,3,2017,1
6,2017-01,ANG MO KIO,3 ROOM,447,ANG MO KIO AVE 10,04 TO 06,68.0,NEW GENERATION,1979,280000.0,2017-01-01,4,6,2017,1
12,2017-01,ANG MO KIO,3 ROOM,235,ANG MO KIO AVE 3,04 TO 06,67.0,NEW GENERATION,1977,295000.0,2017-01-01,4,6,2017,1
22,2017-01,ANG MO KIO,3 ROOM,432,ANG MO KIO AVE 10,10 TO 12,67.0,NEW GENERATION,1979,325000.0,2017-01-01,10,12,2017,1
24,2017-01,ANG MO KIO,3 ROOM,584,ANG MO KIO AVE 3,04 TO 06,67.0,NEW GENERATION,1979,328000.0,2017-01-01,4,6,2017,1


In [597]:
df[10:13]            # rows 10 to 13  

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_year,resale_price,transaction_date,min_floor,max_floor,transaction_year,transaction_month
49,2017-01,ANG MO KIO,5 ROOM,101,ANG MO KIO AVE 3,01 TO 03,117.0,STANDARD,1978,500000.0,2017-01-01,1,3,2017,1
53,2017-01,ANG MO KIO,5 ROOM,588A,ANG MO KIO ST 52,10 TO 12,112.0,DBSS,2011,770000.0,2017-01-01,10,12,2017,1
54,2017-01,ANG MO KIO,5 ROOM,310A,ANG MO KIO AVE 1,10 TO 12,121.0,IMPROVED,2012,832000.0,2017-01-01,10,12,2017,1


In [598]:
df[49995:49998]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_year,resale_price,transaction_date,min_floor,max_floor,transaction_year,transaction_month
218661,2025-02,YISHUN,EXECUTIVE,723,YISHUN ST 71,07 TO 09,146.0,MAISONETTE,1986,818888.0,2025-02-01,7,9,2025,2
218663,2025-07,YISHUN,EXECUTIVE,746,YISHUN ST 72,10 TO 12,162.0,ADJOINED FLAT,1984,1245000.0,2025-07-01,10,12,2025,7
218672,2025-09,YISHUN,EXECUTIVE,834,YISHUN ST 81,04 TO 06,146.0,MAISONETTE,1988,990000.0,2025-09-01,4,6,2025,9


## Output Clean Staging CSV

This is the file that will ingest into MySQL.

In [601]:
df_out = df[[
    "transaction_date",
    "town",
    "street_name",
    "block",
    "flat_type",
    "flat_model",
    "storey_range",
    "min_floor",
    "max_floor",
    "floor_area_sqm",
    "lease_commence_year",
    "resale_price"
]]

df_out.to_csv("resale_cleaned.csv", index=False)